# Class Competition

# Who survived the sinking of the Titanic?

The goal of this competition is to predict who survived the Titanic sinking in 1912.

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("Titanic_0.csv")

In [6]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q


In [7]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

## Data set description

<ul>
<li><b>Survived</b>: binary attribute that indicates whether the passenger survived. This is the dependent variable that we will attempt to explain
<li><b>Pclass</b>: Ticket class (1 = 1st class, 2 = 2nd class, 3 = 3rd class)
<li><b>Age</b>: Passenger age
<li><b>SibSp</b>: The amout of the passenger's siblings/spouses aboard the Titanic
<li><b>Parch</b>: The amout of the passenger's parents/children aboard the Titanic
<li><b>Fare</b>: The ticket fare
<li><b>Male</b>: binary attibute that indicates the gender (1=Male, 0=Female)
<li><b>Embarked_C</b>: binary attibute that indicates whether the passenger embarked in Cherbourg
<li><b>Embarked_Q</b>: binary attibute that indicates whether the passenger embarked in Queenstown
<li><b>Embarked_S</b>: binary attibute that indicates whether the passenger embarked in Southampton
</ul>

## Instruction

Cleaning the data set if necessary. 

Use everything you know to find a machine learning model to achieve the highest possible AUC score. Two testing sets have been reserved: TestA.csv and TestB.csv. Your model will be evaluated using these two sets. 70% of the grade will be based on the AUC score on TestA.csv. 30% of the grade will be based on the ranking of the AUC score on TestB.csv among the groups. To be specific, your grade on TestA.csv will be equal to the final AUC score multiplied by 70, and your grade on TestB.csv will be equal to 30 * (number of groups - your ranking)/(number of groups - 1). You must submit the same model for both sets with clear explanation of your codes. You must include the codes to evaluate your model on TestA.csv and TestB.csv. Failure to do so will result in 20% loss of grades (10% for each test). 

TestB.csv is private, which means you will never see it. The ranking will be revealed only after the deadline. TestA.csv is semi-private. This means that you have at most one chance everyday for me to check your model performance on TestA.csv using your code, and I will let you know the AUC score and post your score on the discussion board. I will save your notebook file in the same folder with the data files. If your code does not work on my computer, you lose the opportunity on the same day. 

In [14]:
# Import Libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
import re
import joblib  # For saving the model

# Load Titanic Dataset
df = pd.read_csv("Titanic_0.csv")
# Feature Engineering
def feature_engineering(data):
    # Family size
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    
    # Extract Title from Name
    data['Title'] = data['Name'].apply(lambda x: re.search(r' ([A-Za-z]+)\.', x).group(1) if re.search(r' ([A-Za-z]+)\.', x) else 'None')
    rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
    data['Title'] = data['Title'].replace(rare_titles, 'Rare')
    data['Title'] = data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    
    # Fare per person
    data['FarePerPerson'] = data['Fare'] / data['FamilySize']
    
    return data
df = feature_engineering(df)

# Define Features and Target
y = df['Survived']
X = df.drop(['Survived', 'Name', 'Ticket', 'Cabin'], axis=1)

# Preprocessing Pipeline
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# logisticRegression with GridSearchCV
pipe_lr = make_pipeline(preprocessor, LogisticRegression(max_iter=1000, random_state=42))
param_grid_lr = {'logisticregression__C': [0.01,0.1, 1.0, 10, 100]}
grid_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=5, scoring= 'roc_auc', n_jobs=-1)
grid_lr.fit(X_train, y_train)
# save the best regression model
best_lr = grid_lr.best_estimator_
# get probability
y_prob_lr = best_lr.predict_proba(X_test)
# roc_auc_score
roc_auc_lr = roc_auc_score(y_test, y_prob_lr[:,1])
#print(f'LogisticRegression ROC AUC Score: {roc_auc_lr:.3f}') #0.871


# randomforest with GridSearchCV
pipe_rf = make_pipeline(preprocessor, RandomForestClassifier(random_state=42))
param_grid_rf = {
    'randomforestclassifier__n_estimators': [100, 200],
    'randomforestclassifier__max_depth': [10, 20, None],
    'randomforestclassifier__min_samples_split': [5, 10]
}
grid_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=5, scoring= 'roc_auc', n_jobs=-1)
grid_rf.fit(X_train, y_train)
# save the best randomforestmodel
best_rf = grid_rf.best_estimator_
# get probability
y_prob_rf = best_rf.predict_proba(X_test)
# roc_auc_score
roc_auc_rf = roc_auc_score(y_test, y_prob_rf[:,1])
#print(f'RandomForest ROC AUC Score: {roc_auc_rf:.3f}') #0.857


# XGBoost with GridSearchCV
pipe_xgb = make_pipeline(preprocessor, XGBClassifier(random_state=42))
param_grid_xgb = {
    'xgbclassifier__max_depth':[3,5,7,9],
    'xgbclassifier__n_estimators':[100,200,300],
    'xgbclassifier__learning_rate':[0.01, 0.1, 0.2]
}
grid_xgb = GridSearchCV(pipe_xgb, param_grid_xgb, cv=5, scoring = 'roc_auc', n_jobs=-1)
grid_xgb.fit(X_train, y_train)
# save the best XGboost
best_xgb = grid_xgb.best_estimator_
# get probability
y_prob_xgb = best_xgb.predict_proba(X_test)
# roc_auc_score
roc_auc_xgb = roc_auc_score(y_test, y_prob_xgb[:,1])
#print(f'XGboost ROC AUC Score: {roc_auc_xgb:.3f}') #0.855

# Save the trained models
joblib.dump(best_lr, "best_lr.pkl")
joblib.dump(best_rf, "best_rf.pkl")
joblib.dump(best_xgb, "best_xgb.pkl")

# Ensemble Model
ensemble_model = VotingClassifier(
    estimators=[
        ('RandomForest', best_rf),
        ('XGBoost', best_xgb),
        ('LogisticRegression', best_lr)
    ],
    voting='soft'
)
# Train the Ensemble Model
ensemble_model.fit(X_train, y_train)
# Evaluate the Model
y_pred_proba = ensemble_model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"Ensemble Model ROC AUC Score (Training): {roc_auc:.4f}")

Ensemble Model ROC AUC Score (Training): 0.8810


In [11]:
# Import Libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
import re
import joblib  # For saving the model

# Load Pre-Trained Models (from "Titanic_0.csv")
best_lr = joblib.load("best_lr.pkl")  # Load trained Logistic Regression
best_rf = joblib.load("best_rf.pkl")  # Load trained Random Forest
best_xgb = joblib.load("best_xgb.pkl")  # Load trained XGBoost

# load data
df = pd.read_csv("testA.csv")  

# Feature Engineering for testA
def feature_engineering(data):
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    data['Title'] = data['Name'].apply(lambda x: re.search(r' ([A-Za-z]+)\.', x).group(1) if re.search(r' ([A-Za-z]+)\.', x) else 'None')
    rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
    data['Title'] = data['Title'].replace(rare_titles, 'Rare')
    data['Title'] = data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    data['FarePerPerson'] = data['Fare'] / data['FamilySize']
    return data

df = feature_engineering(df)

# Drop unnecessary columns
df = df.drop(['Name', 'Ticket', 'Cabin'], axis=1)

# Define Features and Target
y_test = df['Survived']  
X_test = df.drop(columns=['Survived']) 

# Make Predictions Using Pre-Trained Models 
y_prob_lr_test = best_lr.predict_proba(X_test)[:, 1]
roc_auc_lr_test = roc_auc_score(y_test, y_prob_lr_test)

y_prob_rf_test = best_rf.predict_proba(X_test)[:, 1]
roc_auc_rf_test = roc_auc_score(y_test, y_prob_rf_test)

y_prob_xgb_test = best_xgb.predict_proba(X_test)[:, 1]
roc_auc_xgb_test = roc_auc_score(y_test, y_prob_xgb_test)

# Print Model AUC Scores
#print(f"Logistic Regression ROC AUC: {roc_auc_lr_test:.4f}")
#print(f"Random Forest ROC AUC: {roc_auc_rf_test:.4f}")
#print(f"XGBoost ROC AUC: {roc_auc_xgb_test:.4f}")

# Ensemble Model 
ensemble_model_test = VotingClassifier(
    estimators=[
        ('RandomForest', best_rf),
        ('XGBoost', best_xgb),
        ('LogisticRegression', best_lr)
    ],
    voting='soft'
)
ensemble_model_test.fit(X_test, y_test)

# Final Predictions on testA.csv (DO NOT FIT AGAIN)
y_pred_proba_test = ensemble_model_test.predict_proba(X_test)[:, 1]
roc_auc_ensemble = roc_auc_score(y_test, y_pred_proba_test)
print(f"✅ FINAL Ensemble Model ROC AUC Score (testA.csv): {roc_auc_ensemble:.4f}")

FileNotFoundError: [Errno 2] No such file or directory: 'testA.csv'

In [ ]:
# Import Libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
import re
import joblib  # For saving the model

# Load Pre-Trained Models (from "Titanic_0.csv")
best_lr = joblib.load("best_lr.pkl")  # Load trained Logistic Regression
best_rf = joblib.load("best_rf.pkl")  # Load trained Random Forest
best_xgb = joblib.load("best_xgb.pkl")  # Load trained XGBoost

# load data
df = pd.read_csv("testB.csv")  

# Feature Engineering for testA
def feature_engineering(data):
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    data['Title'] = data['Name'].apply(lambda x: re.search(r' ([A-Za-z]+)\.', x).group(1) if re.search(r' ([A-Za-z]+)\.', x) else 'None')
    rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
    data['Title'] = data['Title'].replace(rare_titles, 'Rare')
    data['Title'] = data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    data['FarePerPerson'] = data['Fare'] / data['FamilySize']
    return data

df = feature_engineering(df)

# Drop unnecessary columns
df = df.drop(['Name', 'Ticket', 'Cabin'], axis=1)

# Define Features and Target
y_test = df['Survived']  
X_test = df.drop(columns=['Survived']) 

# Make Predictions Using Pre-Trained Models 
y_prob_lr_test = best_lr.predict_proba(X_test)[:, 1]
roc_auc_lr_test = roc_auc_score(y_test, y_prob_lr_test)

y_prob_rf_test = best_rf.predict_proba(X_test)[:, 1]
roc_auc_rf_test = roc_auc_score(y_test, y_prob_rf_test)

y_prob_xgb_test = best_xgb.predict_proba(X_test)[:, 1]
roc_auc_xgb_test = roc_auc_score(y_test, y_prob_xgb_test)

# Print Model AUC Scores
#print(f"Logistic Regression ROC AUC: {roc_auc_lr_test:.4f}")
#print(f"Random Forest ROC AUC: {roc_auc_rf_test:.4f}")
#print(f"XGBoost ROC AUC: {roc_auc_xgb_test:.4f}")

# Ensemble Model 
ensemble_model_test = VotingClassifier(
    estimators=[
        ('RandomForest', best_rf),
        ('XGBoost', best_xgb),
        ('LogisticRegression', best_lr)
    ],
    voting='soft'
)
ensemble_model_test.fit(X_test, y_test)

# Final Predictions on testA.csv (DO NOT FIT AGAIN)
y_pred_proba_test = ensemble_model_test.predict_proba(X_test)[:, 1]
roc_auc_ensemble = roc_auc_score(y_test, y_pred_proba_test)
print(f"✅ FINAL Ensemble Model ROC AUC Score (testB.csv): {roc_auc_ensemble:.4f}")